# Model 1: 10 training observations

Run this notebook independently from top to bottom. It includes the dummy baseline and the first text classifier.

In [ ]:
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

SEED = 42
wine = pd.read_parquet("df_with_color.parquet")
print("Loaded reviews:", len(wine))

## Select the subset and create separate samples


In [ ]:
expected_colors = {"Cabernet Sauvignon": "red", "Chardonnay": "white"}
subset = wine.loc[
    wine["province"].eq("California")
    & wine["variety"].isin(expected_colors)
].dropna(subset=["description", "color"]).copy()
subset["description_key"] = (
    subset["description"].str.replace(r"\s+", " ", regex=True).str.strip().str.casefold()
)
subset = subset.loc[subset["description_key"].ne("")].copy()
assert subset["color"].eq(subset["variety"].map(expected_colors)).all(), "Check color mapping."
conflicting = subset.groupby("description_key")["color"].nunique()
subset = subset.loc[~subset["description_key"].isin(conflicting[conflicting > 1].index)]
subset = subset.drop_duplicates("description_key").reset_index(drop=True)

training_parts, validation_parts = [], []
for color in ["red", "white"]:
    pool = subset.loc[subset["color"].eq(color)]
    assert len(pool) >= 55, f"Not enough {color} observations."
    selected = pool.sample(n=55, random_state=SEED)
    training_parts.append(selected.iloc[:5])
    validation_parts.append(selected.iloc[5:])

train = pd.concat(training_parts).sample(frac=1, random_state=SEED).copy()
validation = pd.concat(validation_parts).sample(frac=1, random_state=SEED).copy()
assert len(train) == 10
assert set(train["description_key"]).isdisjoint(validation["description_key"])
print("Training counts:\n", train["color"].value_counts())
print("\nValidation counts:\n", validation["color"].value_counts())

## Inspect the ten training observations



In [ ]:
for number, (_, row) in enumerate(train.iterrows(), 1):
    print(f"{number}. {row['color']} | {row['variety']}")
    print("Title:", row.get("title"))
    print("Designation:", row.get("designation"))
    print("Review:", row["description"])
    print()

## Fit a baseline and a simple text model


In [ ]:
X_train, y_train = train["description"], train["color"]
X_val, y_val = validation["description"], validation["color"]

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)

model = make_pipeline(
    TfidfVectorizer(max_features=100, ngram_range=(1, 1), min_df=1),
    LogisticRegression(C=1.0, max_iter=1000, random_state=SEED),
)
model.fit(X_train, y_train)
print("Reviews used to fit the model:", len(X_train))
print("TF-IDF vocabulary size:", len(model[0].get_feature_names_out()))

In [ ]:
rows = []
for name, estimator in [("Majority baseline", baseline), ("TF-IDF + logistic regression", model)]:
    for split, X, y in [("training", X_train, y_train), ("validation", X_val, y_val)]:
        predicted = estimator.predict(X)
        rows.append({
            "model": name, "split": split, "n": len(y),
            "accuracy": accuracy_score(y, predicted),
            "macro_f1": f1_score(y, predicted, average="macro", zero_division=0),
        })
results = pd.DataFrame(rows)
print(results.to_string(index=False))
predictions = model.predict(X_val)
print("\nValidation classification report:")
print(classification_report(y_val, predictions, digits=3, zero_division=0))
print("Confusion matrix: rows = actual, columns = predicted")
print(pd.DataFrame(confusion_matrix(y_val, predictions, labels=["red", "white"]),
                   index=["actual red", "actual white"],
                   columns=["predicted red", "predicted white"]))

## Inspect the learned words and errors

In [ ]:
classifier = model[1]
weights = pd.Series(classifier.coef_[0], index=model[0].get_feature_names_out())
print("Classes:", classifier.classes_)
print(f"\nWords favoring {classifier.classes_[0]}:")
print(weights.nsmallest(10).to_string())
print(f"\nWords favoring {classifier.classes_[1]}:")
print(weights.nlargest(10).to_string())

errors = validation.assign(predicted=predictions)
errors = errors.loc[errors["color"].ne(errors["predicted"])]
print("\nValidation mistakes:", len(errors))
for _, row in errors.head(5).iterrows():
    print(f"Actual: {row['color']} | Predicted: {row['predicted']}")
    print(row["description"])
    print()

## Experiment log and next decision

The dummy baseline always predicts one color and achieves 50% accuracy on these balanced samples.

Model 1 uses TF-IDF plus logistic regression trained on ten reviews. The observed training accuracy was 100% and validation accuracy was 78%. Thus, this suggests overfitting. Next, we're going to increase the training data while keeping model settings and validation reviews fixed.